In [12]:
import psycopg2
from dotenv import load_dotenv, find_dotenv
import os
import pandas as pd

load_dotenv(find_dotenv())

True

In [13]:
# Create a psycopg2 connection
conn = psycopg2.connect(
    host=os.environ.get("HOST"),
    database=os.environ.get("DBNAME"),
    user=os.environ.get("USER"),
    password=os.environ.get("PASSWORD"),
    port="5432",
)

In [14]:
# Create a cursor
cursor = conn.cursor()

In [15]:
# run a sql query
query = """
    SELECT "participant_email", "participant_name", SUM("score") AS total_scores,    
        SUM("total") as out_of_scores,    
        (SUM("score")/SUM("total"))*100 as scores_in_percentage    
        FROM "skills_fact" sk  
        join orders_dimension od   
        on sk.order_id =od.id  
        and od.description like '%FRAC-IMG-09102023%'  
        WHERE "project_name" = 'SQL Assessment' AND "is_current" = True    
        GROUP BY "participant_email", "participant_name"  
        ORDER BY scores_in_percentage DESC  
        LIMIT 5; 
"""
cursor.execute(query)
result = cursor.fetchall()
columns = [desc[0] for desc in cursor.description] # fetch all the columns from the table

In [17]:
skills_df = pd.DataFrame(result, columns=columns)
skills_df.head()

,participant_email,participant_name,total_scores,out_of_scores,scores_in_percentage
0,vaishnavi.nimmaturi@fractal.ai,Vaishnavi Nimmaturi,182.0,200.0,91.0
1,parth.mohril@fractal.ai,Parth Ramakant Mohril,180.0,200.0,90.0
2,ishita.gupta@fractal.ai,Ishita Gupta,180.0,200.0,90.0
3,akrati.rawat@fractal.ai,Akrati Rawat,179.0,200.0,89.5
4,minautee.m@fractal.ai,Minautee Minautee,178.0,200.0,89.0


In [18]:
skills_df.columns

Index(['participant_email', 'participant_name', 'total_scores',
       'out_of_scores', 'scores_in_percentage'],
      dtype='object')